In [1]:
import os
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import hitop_cfa
from hitop_cfa.r_env import ro, localconverter, pandas2ri
from hitop_cfa import (WU_ESTABROOK_LEVELS, fit_measeq_level, set_seeds,
                       exhaustive_search_scale, stepwise_configural_cap)
from hitop_cfa.measeq import fit_is_converged
from hitop_cfa.fit import push_model_to_r
from hitop_cfa.scales import OTHER_SCALES, ORIG_ITEMS

In [2]:
log_dir = Path('./log'); log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa_conventional_criteria'
cfa_dir.mkdir(exist_ok=True)
helpfile_dir = cfa_dir / 'temp'; helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'

In [3]:
def get_architecture():
    arch = platform.machine().lower()
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    return "x86"

total_cpus = os.cpu_count()
cpus_to_use = total_cpus - 2 if get_architecture() == 'ARM' else total_cpus // 2 - 1
print(f"Using {cpus_to_use} CPUs for permutation tests")

num_iter = 1000
exh_min_items = 3   # section D floor (sandbox overrides may raise this)

Using 14 CPUs for permutation tests


In [4]:
set_seeds(12345)

In [5]:
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'

# grid1st data, concat order identical to cfa_other_gp_en (gp first):
# permutation draws depend on row order, so this keeps every test here --
# and the stepwise configural certificate used in section D -- seed-valid
# load
data_val = pd.read_csv(path_save_val)
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])
datasets_stepwise = {'val_gp': data_val_genpop, 'val_en': data_val_enriched, 'gp_en': data_genpop_enriched}

PHQ_FORMULA = OTHER_SCALES['phq_sum']
PHQ_ITEMS = PHQ_FORMULA.split('=~', 1)[1].strip().split(' + ')
print(f"gp n={len(data_gp)}, en n={len(data_en)}; PHQ items: {PHQ_ITEMS}")

gp n=497, en n=305; PHQ items: ['phq_1', 'phq_2', 'phq_3', 'phq_4', 'phq_5', 'phq_6', 'phq_7', 'phq_8']


In [6]:
ALL_SCALE_ITEMS = list(OTHER_SCALES.items()) + list(ORIG_ITEMS.items())

lax_rows = []
for pair, data in datasets_stepwise.items():
    # PHQ/GAD/BAARS were not administered in the validation sample, so the
    # other scales are screenable on the gp_en pair only
    if pair == 'gp_en':
        scale_items = ALL_SCALE_ITEMS
    else:
        scale_items = list(ORIG_ITEMS.items())
    for scale, formula in scale_items:
        item_list = formula.split('=~', 1)[1].strip().split(' + ')
        push_model_to_r(scale, item_list, data, path_to_helpfile)
        prev = None
        for level, ge in WU_ESTABROOK_LEVELS:
            fit_measeq_level(item_list, ge, r_fit_name=f'fit_{level}')
            if not fit_is_converged(f'fit_{level}'):
                lax_rows.append(dict(pair=pair, scale=scale, level=level,
                                     converged=False))
                break
            cfi, rmsea, chisq, df = (float(v) for v in ro.r(
                f'lavaan::fitMeasures(fit_{level}, '
                f'c("cfi.scaled", "rmsea.scaled", "chisq.scaled", "df"))'))
            row = dict(pair=pair, scale=scale, level=level, converged=True,
                       cfi=round(cfi, 4), rmsea=round(rmsea, 4),
                       chisq_scaled=round(chisq, 2), df=int(df))
            if prev is not None and pd.notna(cfi) and pd.notna(prev['cfi']):
                row['delta_cfi'] = round(cfi - prev['cfi'], 4)
                row['delta_rmsea'] = round(rmsea - prev['rmsea'], 4)
                row['passes_dCFI'] = row['delta_cfi'] >= -0.010
                row['passes_dRMSEA'] = row['delta_rmsea'] <= 0.015
            lax_rows.append(row)
            prev = row
lax = pd.DataFrame(lax_rows)
lax.to_csv(cfa_dir / 'laxer_criteria_screen.csv', index=False)
lax

,pair,scale,level,converged,cfi,rmsea,chisq_scaled,df,delta_cfi,delta_rmsea,passes_dCFI,passes_dRMSEA
0,val_gp,anhedonic_depression,configural,True,0.9757,0.1038,443.88,70,NaN,NaN,NaN,NaN
1,val_gp,anhedonic_depression,thresholds,True,0.9739,0.1006,481.47,80,-0.0018,-0.0032,True,True
2,val_gp,anhedonic_depression,metric,True,0.9755,0.0925,466.49,89,0.0016,-0.0081,True,True
3,val_gp,anhedonic_depression,scalar,True,0.9747,0.0895,487.03,98,-0.0008,-0.0030,True,True
4,val_gp,anhedonic_depression,strict,True,0.9793,0.0772,427.16,108,0.0046,-0.0123,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
235,gp_en,well_being,configural,True,0.9721,0.1185,463.41,70,NaN,NaN,NaN,NaN
236,gp_en,well_being,thresholds,True,0.9709,0.1134,491.21,80,-0.0012,-0.0051,True,True
237,gp_en,well_being,metric,True,0.9747,0.1002,446.11,89,0.0038,-0.0132,True,True
238,gp_en,well_being,scalar,True,0.9750,0.0948,450.36,98,0.0003,-0.0054,True,True
